# Fase 4 — Las 10 operaciones con Modin

**Proyecto:** BigData-Proy_Ciberseguridad · CIC-IoT-2023  
**Fase:** 4 — Procesamiento distribuido  
**Motor:** Modin

**Modin** no es un motor de cálculo: es una capa que expone la API de pandas y delega el trabajo real. En este notebook se ejecuta con **backend Dask**, de modo que el mismo código con `import pandas` se quedaría en un solo núcleo.

| Dato | Valor |
|---|---|
| Dataset | `01_fase1_datos/muestra/CICIoT2023_sample_600k.csv` |
| Registros | 600,000 |
| Columnas | 40 (39 numéricas + `Label`) |
| Clases | 34 (1 benigna + 33 de ataque) |
| Modo de ejecución | local, en la computadora del proyecto |
| Salida | `results/fase4/Modin/` |

Este notebook ejecuta las 10 operaciones del enunciado con **Modin** y guarda
cada resultado como CSV para poder compararlo con los demás motores.

## Las 10 operaciones del enunciado

| # | Operación | Requisito del enunciado | Archivo de salida |
|---|---|---|---|
| 01 | Carga y validación del dataset | Validación | `01_validacion.csv` |
| 02 | Limpieza de valores no válidos | Limpieza de datos | `02_limpieza.csv` |
| 03 | Tratamiento de duplicados | Eliminación de duplicados | `03_duplicados.csv` |
| 04 | Transformación de variables | Transformación de variables | `04_transformacion_variables.csv` |
| 05 | Filtrado de tráfico | Filtrado | `05_filtrado.csv` |
| 06 | Agregaciones globales | Agregaciones | `06_agregaciones.csv` |
| 07 | Agrupaciones por clase y protocolo | Agrupaciones | `07_agrupaciones.csv` |
| 08 | Ordenamiento y Top-10 | Ordenamiento | `08_ordenamiento_top10.csv` |
| 09 | Métricas de ciberseguridad | Cálculo de métricas | `09_metricas_ciberseguridad.csv` |
| 10 | Resumen consolidado por clase | CRUD y tablas de resultado | `10_resumen_consolidado.csv` |

## Reglas comunes a los cuatro motores

Para que la comparación sea justa, los cuatro notebooks aplican exactamente las
mismas reglas:

1. **Redondeo de ingesta.** Cada motor usa un parser de CSV distinto y los últimos
   decimales de un float pueden diferir en torno a `1e-11`. Todos redondean a
   **6 decimales** al leer el archivo.
2. **Valididad.** Una celda es *no válida* si está vacía, es `NaN` o es infinita.
   La operación 02 elimina las filas que contienen alguna celda no válida.
3. **Extremos.** `minimo_valido` y `maximo_valido` se calculan solo sobre valores
   finitos.
4. **Umbrales.** Los percentiles salen de `04_fase4_procesamiento/comun/umbrales.json`,
   calculado una vez con la biblioteca estándar, para que el filtro sea idéntico.
5. **Escritura.** Todos los CSV se escriben con el mismo formateador
   (`comun/io_comun.py`), así que se comparan celda por celda.

## Preparación del entorno

In [1]:
from __future__ import annotations

import sys
import time
from importlib.metadata import version
from pathlib import Path

AQUI = Path.cwd()
FASE = AQUI if (AQUI / "comun").exists() else AQUI.parent
if str(FASE) not in sys.path:
    sys.path.insert(0, str(FASE))

from comun import config
from comun.io_comun import escribir_csv, escribir_json, pct

import numpy as np
import modin.config

# Modin es la API; Dask es el motor que reparte el trabajo.
modin.config.Engine.put("dask")

import modin.pandas as pd

MOTOR = "modin"
VERSION = version("modin")
CSV_MUESTRA = config.CSV_MUESTRA
SALIDA = config.RUTA_RESULTADOS / MOTOR
SALIDA.mkdir(parents=True, exist_ok=True)
UMBRALES = config.cargar_umbrales()
TIEMPOS: list[dict] = []

print(f"Modin       {VERSION}")
print(f"Backend     {modin.config.Engine.get()}")
print(f"Python      {sys.version.split()[0]}")
print(f"Dataset     {CSV_MUESTRA.name} ({CSV_MUESTRA.stat().st_size:,} bytes)")
print(f"Resultados  {SALIDA}")


def medir(id_operacion: str, nombre: str, funcion):
    """Ejecuta la operación, cronometra y registra el tiempo."""
    inicio = time.perf_counter()
    resultado = funcion()
    segundos = round(time.perf_counter() - inicio, 3)
    TIEMPOS.append(
        {
            "motor": MOTOR,
            "id_operacion": id_operacion,
            "operacion": nombre,
            "segundos": segundos,
        }
    )
    print(f"  -> [{id_operacion}] {nombre}: {segundos:.3f} s")
    return resultado


def csv(nombre: str, columnas: list[str], registros) -> None:
    """Escribe un resultado con el formateador común a los cuatro motores."""
    escribir_csv(SALIDA / nombre, columnas, registros)


def json_(nombre: str, contenido) -> None:
    escribir_json(SALIDA / nombre, contenido)

Modin       0.37.1
Backend     Dask
Python      3.12.3
Dataset     CICIoT2023_sample_600k.csv (124,498,206 bytes)
Resultados  C:\Users\steve\Desktop\Big_Data\ProyectoBigData_CICIoT2023\results\fase4\modin


## Carga del dataset

Misma regla que en los otros motores: leer el CSV y redondear las columnas decimales a 6 decimales.

In [2]:
def cargar():
    """Lee el CSV con la API de pandas y normaliza la precisión."""
    datos = pd.read_csv(CSV_MUESTRA)
    return datos.round(config.REDONDEO_INGESTA)


df = medir("00", "Carga del dataset", cargar)
print(f"Filas: {len(df):,}   Columnas: {df.shape[1]}")
df.head(5)

  -> [00] Carga del dataset: 9.262 s


Filas: 600,000   Columnas: 40


,Header_Length,Protocol Type,Time_To_Live,Rate,fin_flag_number,syn_flag_number,rst_flag_number,psh_flag_number,ack_flag_number,ece_flag_number,...,Tot sum,Min,Max,AVG,Std,Tot size,IAT,Number,Variance,Label
0,0.00,47,64.00,2109.927612,0.0,0.0,0.0,0.0,0.0,0.0,...,57800,578,578,578.00,0.000000,578.00,0.000475,100,0.000000,Mirai-greip_flood
1,0.00,47,64.00,5564.803906,0.0,0.0,0.0,0.0,0.0,0.0,...,57800,578,578,578.00,0.000000,578.00,0.000181,100,0.000000,Mirai-greip_flood
2,0.16,47,65.91,1789.707156,0.0,0.0,0.0,0.0,0.0,0.0,...,56464,60,578,564.64,77.409383,564.64,0.000560,100,5992.212525,Mirai-greip_flood
3,0.00,47,64.00,2923.022886,0.0,0.0,0.0,0.0,0.0,0.0,...,57800,578,578,578.00,0.000000,578.00,0.000400,100,0.000000,Mirai-greip_flood
4,0.32,47,63.36,600.243572,0.0,0.0,0.0,0.0,0.0,0.0,...,54588,60,578,545.88,119.524502,545.88,0.001667,100,14286.106667,Mirai-greip_flood


## Operación 01 — Carga y validación del dataset

**Requisito del enunciado:** Validación

Mismo recorrido por las 40 columnas: tipo normalizado, celdas no válidas, valores distintos y rango de los valores finitos.

In [3]:
def _tipo(serie) -> str:
    """Tipo normalizado para que los cuatro motores coincidan."""
    if serie.dtype == object:
        return "texto"
    if pd.api.types.is_integer_dtype(serie):
        return "entero"
    if pd.api.types.is_float_dtype(serie):
        return "decimal"
    return "otro"


def _no_validas(serie, tipo: str) -> int:
    """Celdas vacías, NaN o infinitas."""
    invalidas = int(serie.isna().sum())
    if tipo == "decimal":
        invalidas += int((serie == np.inf).sum()) + int((serie == -np.inf).sum())
    return invalidas


def _rango_valido(serie, tipo: str):
    """Mínimo y máximo sobre valores finitos."""
    if tipo == "texto":
        return "", ""
    if tipo == "decimal":
        serie = serie[np.isfinite(serie)]
    return serie.min(), serie.max()


def _tabla(registros):
    """Vista legible de una lista de diccionarios."""
    return pd.DataFrame(registros)


def op01():
    registros = []
    for columna in config.COLUMNAS:
        serie = df[columna]
        tipo = _tipo(serie)
        invalidas = _no_validas(serie, tipo)
        minimo, maximo = _rango_valido(serie, tipo)
        registros.append(
            {
                "columna": columna,
                "tipo": tipo,
                "celdas_no_validas": invalidas,
                "pct_no_validas": round(pct(invalidas, len(df)), 6),
                "distintos": int(serie.nunique(dropna=False)) if columna in config.COLS_DISTINTOS else "",
                "minimo_valido": minimo,
                "maximo_valido": maximo,
            }
        )
    csv("01_validacion.csv",
        ["columna", "tipo", "celdas_no_validas", "pct_no_validas", "distintos",
         "minimo_valido", "maximo_valido"],
        registros)
    json_("01_validacion_resumen.json",
          {"motor": MOTOR, "backend": modin.config.Engine.get(),
           "filas": len(df), "columnas": len(config.COLUMNAS),
           "redondeo_decimales_ingesta": config.REDONDEO_INGESTA,
           "nulos_totales": int(df.isna().sum().sum()),
           "etiquetas_distintas": int(df["Label"].nunique())})
    return registros


validacion = medir("01", "Carga y validación del dataset", op01)
_tabla(validacion).head(12)

Please refer to https://modin.readthedocs.io/en/stable/supported_apis/defaulting_to_pandas.html for explanation.


  -> [01] Carga y validación del dataset: 161.104 s


,columna,tipo,celdas_no_validas,pct_no_validas,distintos,minimo_valido,maximo_valido
0,Header_Length,decimal,0,0.000000,,0.0,60.0
1,Protocol Type,entero,0,0.000000,5,0,47
2,Time_To_Live,decimal,0,0.000000,,0.0,255.0
3,Rate,decimal,13,0.002167,99172,0.000428,7340032.0
4,fin_flag_number,decimal,0,0.000000,110,0.0,1.0
5,syn_flag_number,decimal,0,0.000000,145,0.0,1.0
6,rst_flag_number,decimal,0,0.000000,120,0.0,1.0
7,psh_flag_number,decimal,0,0.000000,143,0.0,1.0
8,ack_flag_number,decimal,0,0.000000,188,0.0,1.0
9,ece_flag_number,decimal,0,0.000000,11,0.0,0.6


## Operación 02 — Limpieza de valores no válidos

**Requisito del enunciado:** Limpieza de datos

Los valores `inf` se reemplazan por `NaN` y después se eliminan las filas con algún dato faltante. Es la forma más natural de expresarlo con la API de pandas.

In [4]:
def op02():
    tipos = {c: _tipo(df[c]) for c in config.COLUMNAS}
    antes = len(df)
    no_validas_antes = {c: _no_validas(df[c], tipos[c]) for c in config.COLUMNAS}

    # inf -> NaN y luego dropna()
    limpio = df.replace([np.inf, -np.inf], np.nan).dropna()
    despues = len(limpio)
    no_validas_despues = {c: _no_validas(limpio[c], tipos[c]) for c in config.COLUMNAS}

    registros = [
        {
            "columna": c,
            "valores_no_validos_antes": no_validas_antes[c],
            "valores_no_validos_despues": no_validas_despues[c],
            "filas_eliminadas": antes - despues,
        }
        for c in config.COLUMNAS
        if no_validas_antes[c] > 0
    ]
    csv("02_limpieza.csv",
        ["columna", "valores_no_validos_antes", "valores_no_validos_despues",
         "filas_eliminadas"],
        registros)
    json_("02_limpieza_resumen.json",
          {"motor": MOTOR,
           "criterio": "se elimina la fila con una celda vacía, NaN o infinita",
           "filas_antes": antes, "filas_despues": despues,
           "filas_eliminadas": antes - despues,
           "columnas_afectadas": registros})
    return limpio


limpio = medir("02", "Limpieza de valores no válidos", op02)
print(f"Filas antes: {len(df):,}   Filas después: {len(limpio):,}   Eliminadas: {len(df) - len(limpio):,}")

  -> [02] Limpieza de valores no válidos: 186.409 s
Filas antes: 600,000   Filas después: 599,987   Eliminadas: 13


## Operación 03 — Tratamiento de duplicados

**Requisito del enunciado:** Eliminación de duplicados

`drop_duplicates()` sin argumentos elimina filas repetidas exactas; con `subset` se trabaja por clave.

In [5]:
COLS_CLAVE = config.COLS_CLAVE_DUPLICADOS


def op03():
    antes = len(limpio)
    exactos = len(limpio.drop_duplicates())
    por_clave = len(limpio.drop_duplicates(subset=COLS_CLAVE))

    registros = [
        {"criterio": "filas_completas",
         "columnas_clave": f"las {len(config.COLUMNAS)} columnas",
         "filas_antes": antes, "filas_despues": exactos,
         "duplicados_eliminados": antes - exactos,
         "pct_duplicados": round(pct(antes - exactos, antes), 6)},
        {"criterio": "columnas_clave",
         "columnas_clave": ", ".join(COLS_CLAVE),
         "filas_antes": antes, "filas_despues": por_clave,
         "duplicados_eliminados": antes - por_clave,
         "pct_duplicados": round(pct(antes - por_clave, antes), 6)},
    ]
    csv("03_duplicados.csv",
        ["criterio", "columnas_clave", "filas_antes", "filas_despues",
         "duplicados_eliminados", "pct_duplicados"],
        registros)

    ejemplos = limpio.groupby(COLS_CLAVE).size().reset_index(name="apariciones")
    ejemplos = ejemplos.sort_values(
        ["apariciones", *COLS_CLAVE], ascending=[False, *([True] * len(COLS_CLAVE))]
    ).head(config.TOP_N)
    csv("03_duplicados_ejemplos.csv", ["apariciones"] + COLS_CLAVE,
        ejemplos.to_dict("records"))
    return registros


duplicados = medir("03", "Tratamiento de duplicados", op03)
_tabla(duplicados)

  -> [03] Tratamiento de duplicados: 16.364 s


,criterio,columnas_clave,filas_antes,filas_despues,duplicados_eliminados,pct_duplicados
0,filas_completas,las 40 columnas,599987,395496,204491,34.082572
1,columnas_clave,"Label, Protocol Type, Tot size, IAT, Rate, Number",599987,345902,254085,42.348418


## Operación 04 — Transformación de variables

**Requisito del enunciado:** Transformación de variables

Las mismas seis variables que en el resto de motores, escritas con `assign` y expresiones condicionales de pandas.

In [6]:
COLS_NUEVAS = [
    "size_kb",
    "rate_mbps",
    "coef_variacion",
    "total_flags",
    "protocolo_principal",
    "rango_iat",
]

DEFINICIONES = {
    "size_kb": "Tot size / 1024 (kilobytes)",
    "rate_mbps": "Rate / 1 000 000 (paquetes por segundo)",
    "coef_variacion": "Std / AVG (dispersion del tamaño de paquete)",
    "total_flags": "Suma de los siete indicadores de flags TCP",
    "protocolo_principal": "Protocolo con valor 1 en las columnas one-hot",
    "rango_iat": "IAT clasificado con los percentiles 50 y 95: bajo, medio, alto",
}

In [7]:
def op04():
    iat = UMBRALES["IAT"]
    trabajo = limpio.copy()

    trabajo["size_kb"] = trabajo["Tot size"] / 1024
    trabajo["rate_mbps"] = trabajo["Rate"] / 1_000_000
    trabajo["coef_variacion"] = np.where(
        trabajo["AVG"] > 0, trabajo["Std"] / trabajo["AVG"].replace(0, np.nan), np.nan
    )
    trabajo["total_flags"] = trabajo[config.COLS_FLAGS].sum(axis=1)
    trabajo["rango_iat"] = np.where(
        trabajo["IAT"] <= iat["p50"], "bajo",
        np.where(trabajo["IAT"] <= iat["p95"], "medio", "alto"),
    )

    # Protocolo principal: la PRIMERA columna one-hot con valor 1, el mismo
    # criterio que usa Polars. argmax devuelve la posicion de ese primer 1.
    banderas = trabajo[config.COLS_PROTOCOLO].to_numpy()
    nombres = np.array(config.COLS_PROTOCOLO, dtype=object)
    trabajo["protocolo_principal"] = np.where(
        banderas.max(axis=1) > 0, nombres[(banderas > 0).argmax(axis=1)], "otro"
    )

    registros = []
    for columna in COLS_NUEVAS:
        serie = trabajo[columna]
        texto = _tipo(serie) == "texto"
        registros.append(
            {
                "columna": columna,
                "tipo": _tipo(serie),
                "nulos": int(serie.isna().sum()),
                "minimo": "" if texto else serie.min(),
                "maximo": "" if texto else serie.max(),
                "media": "" if texto else serie.mean(),
                "distintos": int(serie.nunique()) if texto else "",
                "definicion": DEFINICIONES[columna],
            }
        )
    csv("04_transformacion_variables.csv",
        ["columna", "tipo", "nulos", "minimo", "maximo", "media", "distintos", "definicion"],
        registros)

    muestra = trabajo[["Label", "Protocol Type", "Rate", "Tot size", "IAT", *COLS_NUEVAS]] \
        .head(config.FILAS_MUESTRA_TRANSFORMACION)
    csv("04_transformacion_muestra.csv", list(muestra.columns), muestra.to_dict("records"))
    return trabajo, registros


transformado, variables = medir("04", "Transformación de variables", op04)
_tabla(variables)

  -> [04] Transformación de variables: 30.188 s


,columna,tipo,nulos,minimo,maximo,media,distintos,definicion
0,size_kb,decimal,0,0.044922,4.645605,0.128457,,Tot size / 1024 (kilobytes)
1,rate_mbps,decimal,0,0.0,7.340032,0.028514,,Rate / 1 000 000 (paquetes por segundo)
2,coef_variacion,decimal,0,0.0,5.916701,0.105852,,Std / AVG (dispersion del tamaño de paquete)
3,total_flags,decimal,0,0.0,2.38,0.613194,,Suma de los siete indicadores de flags TCP
4,protocolo_principal,texto,0,,,,13,Protocolo con valor 1 en las columnas one-hot
5,rango_iat,texto,0,,,,3,IAT clasificado con los percentiles 50 y 95: b...


In [8]:
# A partir de aqui df pasa a ser el dataset limpio y transformado:
# las operaciones 05 a 10 ya pueden usar las columnas nuevas.
df = transformado
print(f"Columnas del dataset transformado: {len(df.columns)}")

Columnas del dataset transformado: 46


## Operación 05 — Filtrado de tráfico

**Requisito del enunciado:** Filtrado

Cuatro filtros con los percentiles de `comun/umbrales.json`, escritos con máscaras booleanas de pandas.

In [9]:
def op05():
    total = len(df)
    rate, size, iat = UMBRALES["Rate"], UMBRALES["Tot size"], UMBRALES["IAT"]

    filtros = [
        ("trafico_alto", f"Rate >= p95 ({rate['p95']:.6f})", rate["p95"],
         df["Rate"] >= rate["p95"]),
        ("paquetes_grandes", f"Tot size >= p95 ({size['p95']:.6f})", size["p95"],
         df["Tot size"] >= size["p95"]),
        ("trafico_intenso",
         f"Rate >= p95 ({rate['p95']:.6f}) y Tot size >= p50 ({size['p50']:.6f})",
         rate["p95"],
         (df["Rate"] >= rate["p95"]) & (df["Tot size"] >= size["p50"])),
        ("iat_reducido", f"0 < IAT <= p95 ({iat['p95']:.6f})", iat["p95"],
         (df["IAT"] > 0) & (df["IAT"] <= iat["p95"])),
    ]

    registros = []
    for nombre, condicion, umbral, mascara in filtros:
        encontradas = int(mascara.sum())
        registros.append(
            {"filtro": nombre, "condicion": condicion, "umbral": umbral,
             "filas_encontradas": encontradas,
             "pct_del_total": round(pct(encontradas, total), 6)}
        )
    csv("05_filtrado.csv",
        ["filtro", "condicion", "umbral", "filas_encontradas", "pct_del_total"],
        registros)

    intenso = df[(df["Rate"] >= rate["p95"]) & (df["Tot size"] >= size["p50"])]
    por_label = (
        intenso.groupby("Label")
        .agg(registros=("Label", "size"),
             volumen_bytes=("Tot size", "sum"),
             media_rate=("Rate", "mean"))
        .reset_index()
        .sort_values(["registros", "Label"], ascending=[False, True])
    )
    csv("05_filtrado_por_label.csv",
        ["Label", "registros", "volumen_bytes", "media_rate"],
        por_label.to_dict("records"))
    return registros


filtrado = medir("05", "Filtrado de tráfico", op05)
_tabla(filtrado)

  -> [05] Filtrado de tráfico: 7.524 s


,filtro,condicion,umbral,filas_encontradas,pct_del_total
0,trafico_alto,Rate >= p95 (63492.340297),63492.340297,30029,5.004942
1,paquetes_grandes,Tot size >= p95 (586.680000),586.680000,31235,5.205946
2,trafico_intenso,Rate >= p95 (63492.340297) y Tot size >= p50 (...,63492.340297,30013,5.002275
3,iat_reducido,0 < IAT <= p95 (0.001070),0.001070,569989,95.000225


## Operación 06 — Agregaciones globales

**Requisito del enunciado:** Agregaciones

Siete métricas por cada columna indicadora, en formato largo.

In [10]:
def op06():
    registros = []
    for columna in config.COLS_INDICADORES:
        serie = df[columna]
        for metrica, valor in (
            ("conteo", serie.count()),
            ("suma", serie.sum()),
            ("media", serie.mean()),
            ("mediana", serie.median()),
            ("minimo", serie.min()),
            ("maximo", serie.max()),
            ("desviacion_estandar", serie.std()),
        ):
            registros.append({"columna": columna, "metrica": metrica, "valor": valor})
    csv("06_agregaciones.csv", ["columna", "metrica", "valor"], registros)
    return registros


agregaciones = medir("06", "Agregaciones globales", op06)
_tabla(agregaciones).head(14)

  -> [06] Agregaciones globales: 33.229 s


,columna,metrica,valor
0,Rate,conteo,5.999870e+05
1,Rate,suma,1.710832e+10
2,Rate,media,2.851448e+04
3,Rate,mediana,2.466222e+04
4,Rate,minimo,4.280000e-04
5,Rate,maximo,7.340032e+06
6,Rate,desviacion_estandar,3.262685e+04
7,Tot size,conteo,5.999870e+05
8,Tot size,suma,7.892257e+07
9,Tot size,media,1.315405e+02


## Operación 07 — Agrupaciones por clase y protocolo

**Requisito del enunciado:** Agrupaciones

`groupby` con agregación con nombre, el estilo más cómodo de pandas.

In [11]:
def op07():
    total = len(df)
    grupos = (
        df.groupby(["Label", "Protocol Type"])
        .agg(
            registros=("Label", "size"),
            volumen_bytes=("Tot size", "sum"),
            media_tot_size=("Tot size", "mean"),
            media_rate=("Rate", "mean"),
            media_iat=("IAT", "mean"),
        )
        .reset_index()
        .sort_values(["Label", "Protocol Type"])
    )
    registros = [
        {
            "Label": fila["Label"],
            "Protocol Type": fila["Protocol Type"],
            "registros": int(fila["registros"]),
            "pct_registros": round(pct(int(fila["registros"]), total), 6),
            "volumen_bytes": fila["volumen_bytes"],
            "media_tot_size": fila["media_tot_size"],
            "media_rate": fila["media_rate"],
            "media_iat": fila["media_iat"],
        }
        for _, fila in grupos.iterrows()
    ]
    csv("07_agrupaciones.csv",
        ["Label", "Protocol Type", "registros", "pct_registros", "volumen_bytes",
         "media_tot_size", "media_rate", "media_iat"],
        registros)
    return registros


agrupaciones = medir("07", "Agrupaciones por clase y protocolo", op07)
_tabla(agrupaciones).head(10)

  -> [07] Agrupaciones por clase y protocolo: 39.799 s


,Label,Protocol Type,registros,pct_registros,volumen_bytes,media_tot_size,media_rate,media_iat
0,Backdoor_Malware,6,28,0.004667,1.013480e+04,361.957143,673.427166,0.026105
1,Backdoor_Malware,17,14,0.002333,2.075000e+03,148.214286,48.065148,0.038464
2,Benign,0,31,0.005167,4.701800e+03,151.670968,221.481694,0.019105
3,Benign,1,3,0.000500,4.088000e+02,136.266667,128.237094,0.018639
4,Benign,6,12885,2.147547,8.392353e+06,651.327377,2901.537006,0.008740
5,Benign,17,1166,0.194338,1.785024e+05,153.089537,152.587589,0.013828
6,BrowserHijacking,6,67,0.011167,3.970520e+04,592.614925,23350.552569,0.015639
7,BrowserHijacking,17,9,0.001500,2.711400e+03,301.266667,259.118631,0.014353
8,CommandInjection,6,54,0.009000,3.390350e+04,627.842593,2927.717130,0.015490
9,CommandInjection,17,16,0.002667,2.167700e+03,135.481250,63.663114,0.029455


## Operación 08 — Ordenamiento y Top-10

**Requisito del enunciado:** Ordenamiento

Clases ordenadas por volumen de tráfico descendente.

In [12]:
def _resumen_por_clase():
    return (
        df.groupby("Label")
        .agg(registros=("Label", "size"),
             volumen_bytes=("Tot size", "sum"),
             media_rate=("Rate", "mean"))
        .reset_index()
        .sort_values(["volumen_bytes", "Label"], ascending=[False, True])
    )


def op08():
    volumen_total = float(df["Tot size"].sum())
    top = _resumen_por_clase().head(config.TOP_N)
    registros = [
        {
            "posicion": posicion,
            "Label": fila["Label"],
            "registros": int(fila["registros"]),
            "volumen_bytes": fila["volumen_bytes"],
            "volumen_mb": fila["volumen_bytes"] / (1024 * 1024),
            "pct_volumen": round(pct(float(fila["volumen_bytes"]), volumen_total), 6),
            "media_rate": fila["media_rate"],
        }
        for posicion, (_, fila) in enumerate(top.iterrows(), start=1)
    ]
    csv("08_ordenamiento_top10.csv",
        ["posicion", "Label", "registros", "volumen_bytes", "volumen_mb",
         "pct_volumen", "media_rate"],
        registros)
    return registros


top10 = medir("08", "Ordenamiento y Top-10", op08)
_tabla(top10)

  -> [08] Ordenamiento y Top-10: 5.587 s


,posicion,Label,registros,volumen_bytes,volumen_mb,pct_volumen,media_rate
0,1,Benign,14085,8.575966e+06,8.178679,10.866303,2667.481157
1,2,Mirai-greeth_flood,12719,7.433445e+06,7.089085,9.418655,5576.009272
2,3,Mirai-udpplain,11423,6.243730e+06,5.954485,7.911209,6132.884517
3,4,DDoS-ICMP_Flood,92356,5.610000e+06,5.350113,7.108233,39957.267678
4,5,Mirai-greip_flood,9642,5.451962e+06,5.199396,6.907988,5055.603642
5,6,DDoS-ICMP_Fragmentation,5804,5.140472e+06,4.902337,6.513311,2973.919964
6,7,DDoS-UDP_Flood,69419,4.218632e+06,4.023201,5.345279,33345.355260
7,8,DDoS-TCP_Flood,57687,3.631696e+06,3.463455,4.601593,32271.151585
8,9,DoS-UDP_Flood,39414,3.516836e+06,3.353916,4.456058,20014.109743
9,10,DDoS-UDP_Fragmentation,3681,3.288975e+06,3.136611,4.167344,2026.716545


## Operación 09 — Métricas de ciberseguridad

**Requisito del enunciado:** Cálculo de métricas

Indicadores de población, volumen, comportamiento de flags, concentración y calidad.

In [13]:
def op09():
    total = len(df)
    benignos = int((df["Label"] == config.ETIQUETA_BENIGNA).sum())
    ataques = total - benignos
    volumen_total = float(df["Tot size"].sum())
    paquetes = float(df["Number"].sum())
    clases = int(df["Label"].nunique())
    clases_ataque = int(
        df[df["Label"] != config.ETIQUETA_BENIGNA]["Label"].nunique()
    )
    por_clase = _resumen_por_clase()
    principal = por_clase.iloc[0]
    top5 = por_clase.head(5)
    media_syn = float(df["syn_flag_number"].mean())
    media_ack = float(df["ack_flag_number"].mean())
    duplicados = total - len(df.drop_duplicates())

    metricas = [
        ("poblacion", "total_registros", total),
        ("poblacion", "total_clases", clases),
        ("poblacion", "clases_de_ataque", clases_ataque),
        ("poblacion", "registros_benignos", benignos),
        ("poblacion", "registros_de_ataque", ataques),
        ("poblacion", "pct_registros_benignos", round(pct(benignos, total), 6)),
        ("poblacion", "pct_registros_ataque", round(pct(ataques, total), 6)),
        ("volumen", "volumen_total_bytes", volumen_total),
        ("volumen", "volumen_total_mb", volumen_total / (1024 * 1024)),
        ("volumen", "paquetes_totales", paquetes),
        ("volumen", "tamano_medio_paquete_bytes",
         volumen_total / paquetes if paquetes else 0.0),
        ("volumen", "tasa_media", float(df["Rate"].mean())),
        ("comportamiento", "media_syn_flag", media_syn),
        ("comportamiento", "media_ack_flag", media_ack),
        ("comportamiento", "media_rst_flag", float(df["rst_flag_number"].mean())),
        ("comportamiento", "media_fin_flag", float(df["fin_flag_number"].mean())),
        ("comportamiento", "ratio_syn_ack", media_syn / media_ack if media_ack else 0.0),
        ("comportamiento", "iat_medio", float(df["IAT"].mean())),
        ("concentracion", "clase_principal", principal["Label"]),
        ("concentracion", "pct_registros_clase_principal",
         round(pct(int(principal["registros"]), total), 6)),
        ("concentracion", "pct_registros_top5",
         round(pct(int(top5["registros"].sum()), total), 6)),
        ("concentracion", "pct_volumen_top5",
         round(pct(float(top5["volumen_bytes"].sum()), volumen_total), 6)),
        ("calidad", "filas_duplicadas", duplicados),
        ("calidad", "pct_filas_duplicadas", round(pct(duplicados, total), 6)),
        ("calidad", "pct_filas_unicas", round(pct(total - duplicados, total), 6)),
    ]
    registros = [{"categoria": c, "metrica": m, "valor": v} for c, m, v in metricas]
    csv("09_metricas_ciberseguridad.csv", ["categoria", "metrica", "valor"], registros)
    return registros


metricas = medir("09", "Métricas de ciberseguridad", op09)
_tabla(metricas)

  -> [09] Métricas de ciberseguridad: 18.689 s


,categoria,metrica,valor
0,poblacion,total_registros,599987
1,poblacion,total_clases,34
2,poblacion,clases_de_ataque,33
3,poblacion,registros_benignos,14085
4,poblacion,registros_de_ataque,585902
5,poblacion,pct_registros_benignos,2.347551
6,poblacion,pct_registros_ataque,97.652449
7,volumen,volumen_total_bytes,78922573.190802
8,volumen,volumen_total_mb,75.266431
9,volumen,paquetes_totales,57290468.0


## Operación 10 — Resumen consolidado por clase

**Requisito del enunciado:** CRUD y tablas de resultado

Tabla final por clase, con conteo, peso porcentual, volumen y medias de las variables transformadas.

In [14]:
def op10():
    total = len(df)
    volumen_total = float(df["Tot size"].sum())
    resumen = (
        df.groupby("Label")
        .agg(
            registros=("Label", "size"),
            volumen_bytes=("Tot size", "sum"),
            media_rate=("Rate", "mean"),
            media_tot_size=("Tot size", "mean"),
            media_size_kb=("size_kb", "mean"),
            media_iat=("IAT", "mean"),
            media_syn=("syn_flag_number", "mean"),
            media_ack=("ack_flag_number", "mean"),
            media_rst=("rst_flag_number", "mean"),
            media_total_flags=("total_flags", "mean"),
            protocolos_distintos=("Protocol Type", "nunique"),
        )
        .reset_index()
        .sort_values(["registros", "Label"], ascending=[False, True])
    )
    registros = [
        {
            "posicion": posicion,
            "Label": fila["Label"],
            "registros": int(fila["registros"]),
            "pct_registros": round(pct(int(fila["registros"]), total), 6),
            "volumen_bytes": fila["volumen_bytes"],
            "pct_volumen": round(pct(float(fila["volumen_bytes"]), volumen_total), 6),
            "media_rate": fila["media_rate"],
            "media_tot_size": fila["media_tot_size"],
            "media_size_kb": fila["media_size_kb"],
            "media_iat": fila["media_iat"],
            "media_syn": fila["media_syn"],
            "media_ack": fila["media_ack"],
            "media_rst": fila["media_rst"],
            "media_total_flags": fila["media_total_flags"],
            "protocolos_distintos": int(fila["protocolos_distintos"]),
        }
        for posicion, (_, fila) in enumerate(resumen.iterrows(), start=1)
    ]
    csv("10_resumen_consolidado.csv",
        ["posicion", "Label", "registros", "pct_registros", "volumen_bytes",
         "pct_volumen", "media_rate", "media_tot_size", "media_size_kb", "media_iat",
         "media_syn", "media_ack", "media_rst", "media_total_flags", "protocolos_distintos"],
        registros)
    return registros


consolidado = medir("10", "Resumen consolidado por clase", op10)
_tabla(consolidado).head(10)

  -> [10] Resumen consolidado por clase: 23.555 s


,posicion,Label,registros,pct_registros,volumen_bytes,pct_volumen,media_rate,media_tot_size,media_size_kb,media_iat,media_syn,media_ack,media_rst,media_total_flags,protocolos_distintos
0,1,DDoS-ICMP_Flood,92356,15.393000,5.610000e+06,7.108233,39957.267678,60.743213,0.059320,0.000108,0.000330,0.001629,0.000048,0.002606,4
1,2,DDoS-UDP_Flood,69419,11.570084,4.218632e+06,5.345279,33345.355260,60.770564,0.059346,0.000063,0.000589,0.002523,0.000074,0.004228,4
2,3,DDoS-TCP_Flood,57687,9.614708,3.631696e+06,4.601593,32271.151585,62.955185,0.061480,0.000061,0.000611,0.002596,0.000057,0.004300,2
3,4,DDoS-PSHACK_FLOOD,52521,8.753690,3.175767e+06,4.023902,32611.295328,60.466610,0.059049,0.000056,0.000368,0.967409,0.031412,1.964750,2
4,5,DDoS-SYN_Flood,52065,8.677688,3.247847e+06,4.115232,28803.489591,62.380613,0.060919,0.000071,0.986430,0.016452,0.008175,1.012184,2
5,6,DDoS-RSTFINFLOOD,51887,8.648021,3.176900e+06,4.025337,33311.555872,61.227279,0.059792,0.000366,0.000336,0.002020,0.995544,1.994087,2
6,7,DDoS-SynonymousIP_Flood,46151,7.692000,2.804484e+06,3.553462,32035.141131,60.767560,0.059343,0.000062,0.996234,0.001560,0.000068,0.998458,2
7,8,DoS-UDP_Flood,39414,6.569142,3.516836e+06,4.456058,20014.109743,89.228085,0.087137,0.060188,0.000858,0.005655,0.000136,0.008965,3
8,9,DoS-TCP_Flood,34265,5.710957,2.162479e+06,2.740000,25445.809057,63.110424,0.061631,0.000708,0.000799,0.009931,0.004834,0.017312,2
9,10,DoS-SYN_Flood,26023,4.337261,1.625435e+06,2.059531,22568.571802,62.461477,0.060998,0.000124,0.953776,0.042410,0.034673,1.033366,3


## Resumen de la ejecución

La tabla muestra el tiempo de cada operación. El total incluye la carga del CSV.

In [15]:
tabla_tiempos = _tabla(TIEMPOS)
total = round(sum(f["segundos"] for f in TIEMPOS), 3)
tabla_tiempos

csv("00_tiempos.csv", ["motor", "id_operacion", "operacion", "segundos"], TIEMPOS)
json_(
    "00_resumen_ejecucion.json",
    {"motor": MOTOR, "backend": modin.config.Engine.get(),
     "dataset": CSV_MUESTRA.name, "filas_cargadas": len(limpio),
     "columnas": len(config.COLUMNAS), "operaciones": 10,
     "segundos_totales": total, "tiempos": TIEMPOS},
)
print(f"Tiempo total de las 10 operaciones: {total:.3f} s")
print(f"Archivos escritos en: {SALIDA}")

Tiempo total de las 10 operaciones: 531.710 s
Archivos escritos en: C:\Users\steve\Desktop\Big_Data\ProyectoBigData_CICIoT2023\results\fase4\modin
